# 蘑菇病害3分类 CNN 检测程序

从训练好的 `mushroom_cnn.pth` 模型加载权重，对蘑菇数据集中的3个类别各随机抽取1张图片进行预测。

**3个类别**:
- Healthy → 健康菌丝
- Single_Infected → 单一感染
- Mixed_Infected → 混合感染

In [ ]:
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms

# ====================== 配置参数（和训练代码保持一致） ======================
test_root = r"D:\LHWYAN\download\environment\Python-AI\Mushroom Disease Dataset (Healthy, Single infected & Mixed infected )"
categories = ["Healthy", "Single_Infected", "Mixed_Infected"]
class_names_cn = ["健康菌丝", "单一感染", "混合感染"]
img_size = 100
sample_num = 1          # 每个类别检测1张
model_path = r"D:\LHWYAN\download\environment\Python-AI\mushroom_cnn.pth"
device = torch.device("cpu")

# 图像预处理
test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

# ====================== 网络定义（必须和训练代码完全相同） ======================
class MushroomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 3)          # 输出3类
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        out = self.classifier(x)
        return out

# ====================== 加载模型 ======================
if not os.path.exists(model_path):
    raise FileNotFoundError(f"模型文件不存在！请先运行蘑菇训练代码生成 mushroom_cnn.pth\n路径：{model_path}")
net = MushroomCNN().to(device)
net.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
net.eval()
print(f"模型加载完成（{model_path}），开始检测\n")

# ====================== 单张图片预测函数 ======================
def predict_image(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tensor = test_transform(img_rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = net(tensor).squeeze()               # shape: [3]
        probs = torch.softmax(logits, dim=0)         # shape: [3]
        pred_idx = torch.argmax(probs).item()
        conf = probs[pred_idx].item()
        pred_label = class_names_cn[pred_idx]
    return img_rgb, pred_label, conf, probs.cpu().numpy()

# ====================== 批量检测：3个文件夹各随机1张 ======================
for cls in categories:
    cls_folder = os.path.join(test_root, cls)
    # 筛选有效图片
    valid_img_list = []
    for filename in os.listdir(cls_folder):
        full_path = os.path.join(cls_folder, filename)
        img = cv2.imread(full_path)
        if img is not None:
            valid_img_list.append((full_path, img))

    actual_sample = min(sample_num, len(valid_img_list))
    if actual_sample == 0:
        print(f"警告：{cls} 文件夹下未找到有效图片，跳过该类别")
        continue
    sample_imgs = random.sample(valid_img_list, k=actual_sample)

    true_label_text = class_names_cn[categories.index(cls)]
    print(f"========== 真实类别：{true_label_text}（文件夹 {cls}，共检测{actual_sample}张） ==========")

    for idx, (img_path, raw_bgr_img) in enumerate(sample_imgs):
        img_rgb, pred_label, conf, probs = predict_image(raw_bgr_img)
        file_name = os.path.basename(img_path)
        # 打印各类别概率详情
        prob_str = " | ".join([f"{class_names_cn[i]}：{probs[i]:.2%}" for i in range(3)])
        print(f"第{idx+1}张 | 文件：{file_name}")
        print(f"  真实：{true_label_text} | 预测：{pred_label} | 置信度：{conf:.4f}")
        print(f"  各类别概率：{prob_str}")

        # 弹出图片窗口展示结果
        plt.figure(figsize=(5, 5))
        plt.imshow(img_rgb)
        plt.title(f"真实：{true_label_text}\n预测：{pred_label} 置信度：{conf:.2%}")
        plt.axis("off")
        plt.show()
    print("\n")

print("全部图片检测完毕！")